<a href="https://colab.research.google.com/github/scao1828/main/blob/main/ElectionDay.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [6]:
import os
import time
import pandas as pd
import requests

def fetch_all_major_individual_donors(
    min_amount: float = 10000.0,
    max_amount: float = 100000.0,
    batch_size: int = 1000,
    app_token: str = None
) -> pd.DataFrame:
    """
    Paginates through the NYC Open Data Campaign Contributions dataset using
    Socrata's $offset parameter and passes an X-App-Token header for elevated rate limits.
    """
    endpoint = "https://data.cityofnewyork.us/resource/rjkp-yttg.json"

    headers = {
        "Accept": "application/json",
        "User-Agent": "NYCCampaignPaginatedAnalyzer/5.0"
    }

    if app_token:
        headers["X-App-Token"] = app_token

    all_records = []
    offset = 0
    page_number = 1

    print(f"Starting query for records between ${min_amount:,.2f} and ${max_amount:,.2f}...")

    while True:
        params = {
            "$select": "c_code, name, strno, strname, city, state, zip, date, amnt, recipname",
            "$where": f"amnt >= {min_amount} AND amnt <= {max_amount} AND (c_code = 'IND' OR c_code IS NULL)",
            "$order": "amnt DESC, date DESC",
            "$limit": batch_size,
            "$offset": offset
        }

        try:
            print(f"Fetching Page {page_number} (Offset: {offset}, Limit: {batch_size})...")
            response = requests.get(endpoint, params=params, headers=headers, timeout=30)

            if response.status_code == 429:
                print("Rate limit reached (HTTP 429). Pausing before retrying...")
                time.sleep(5)
                continue

            response.raise_for_status()
            batch = response.json()

            if not batch:
                print("No additional records found. Dataset query complete.")
                break

            all_records.extend(batch)
            retrieved_count = len(batch)
            print(f"  -> Retrieved {retrieved_count} records. Total collected: {len(all_records):,}")

            if retrieved_count < batch_size:
                print("Final batch reached.")
                break

            offset += batch_size
            page_number += 1

        except requests.exceptions.RequestException as err:
            print(f"Error on Page {page_number}: {err}")
            break

    if not all_records:
        return pd.DataFrame()

    df = pd.DataFrame(all_records)
    if 'amnt' in df.columns:
        df['amnt'] = pd.to_numeric(df['amnt'], errors='coerce')

    return df

if __name__ == "__main__":
    # Get token from environment or set as string (optional)
    SOCRATA_TOKEN = os.getenv("SOCRATA_APP_TOKEN")

    donors_df = fetch_all_major_individual_donors(
        min_amount=10000.0,
        max_amount=100000.0,
        batch_size=1000,
        app_token=SOCRATA_TOKEN
    )

    if not donors_df.empty:
        output_filename = "all_nyc_major_individual_donors.csv"
        donors_df.to_csv(output_filename, index=False)
        print(f"\nSaved {len(donors_df):,} records to '{output_filename}'.")

        # Automatically trigger browser download in Google Colab
        try:
            from google.colab import files
            print("Triggering automatic file download in Colab...")
            files.download(output_filename)
        except ImportError:
            print("Not running in Google Colab environment. File is saved locally in working directory.")

Starting query for records between $10,000.00 and $100,000.00...
Fetching Page 1 (Offset: 0, Limit: 1000)...
  -> Retrieved 625 records. Total collected: 625
Final batch reached.

Saved 625 records to 'all_nyc_major_individual_donors.csv'.
Triggering automatic file download in Colab...


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [7]:
import pandas as pd

df = pd.read_csv('all_nyc_major_individual_donors.csv')

# Sort donors by total amount donated ascending to see smaller major donors, or check count of donations
donor_summary = df.groupby(['name', 'city', 'state'])['amnt'].agg(['count', 'sum']).reset_index()

# Sort by sum ascending
print(donor_summary.sort_values(by=['sum', 'count'], ascending=[True, True]).head(30))

                  name           city state  count    sum
0        ALTHEIM, Marc   Lake Success    NY      1  10000
1      Abramson, Leigh       New York    NY      1  10000
6          Ahmad, Ijaz   Old Westbury    NY      1  10000
12       Anand, Shelly        Decatur    GA      1  10000
13      Anderman, Drew      Manhattan    NY      1  10000
21         Barkan, Ady  Santa Barbara    CA      1  10000
24  Bell II, Richard H        Sanibel    FL      1  10000
26    Ben Abraham, Uzi       New York    NY      1  10000
27        Bender, Neil       New York    NY      1  10000
29      Bernstein, Mem       New York    NY      1  10000
30     Betesh, Michael       Brooklyn    NY      1  10000
36       Bibi, Elliott       Brooklyn    NY      1  10000
41          Blau, Lisa       New York    NY      1  10000
50     Brodsky, Daniel       New York    NY      1  10000
53       Cahill, Kelly          Coram    NY      1  10000
65         Chung, Youn       Flushing    NY      1  10000
67    Cohen, A

In [8]:
# Let's inspect a few distinct names in the dataset that do not return widespread public figures (unlike Bloomberg, Tisch, Loeb, Lauder, Walton, etc.)
# We will check donors with modest totals or specific regional profiles.

lesser_known_candidates = [
    'Rehan Muneeb-Azhar',
    'Marc Altheim',
    'Shelly Anand',
    'Stephen J Pajcic III',
    'Drew Anderman'
]

matching_rows = df[df['name'].str.contains('|'.join(['Muneeb-Azhar', 'Altheim', 'Anand', 'Pajcic', 'Anderman']), na=False, case=False)]
print(matching_rows[['name', 'city', 'state', 'amnt', 'recipname']].to_string())

                      name          city state    amnt                                      recipname
1    Pajcic III, Stephen J  Jacksonville    FL  100000                    New Yorkers for Lower Costs
119    Muneeb-Azhar, Rehan  Morton Grove    IL   50000                    New Yorkers for Lower Costs
120    Muneeb-Azhar, Rehan  Morton Grove    IL   50000                    New Yorkers for Lower Costs
261    Muneeb-Azhar, Rehan  Morton Grove    IL   25000                    New Yorkers for Lower Costs
265    Muneeb-Azhar, Rehan  Morton Grove    IL   25000                    New Yorkers for Lower Costs
454          Altheim, Marc  Lake Success    NY   10000                             Fix the City, Inc.
479          Anand, Shelly       Decatur    GA   10000                    New Yorkers for Lower Costs
502          ALTHEIM, Marc  Lake Success    NY   10000  NEW YORKERS FOR A BETTER FUTURE MAYOR 25, INC
551          Altheim, Marc  Lake Success    NY   10000                            